# 05 — Fine-tuning and LoRA, implemented directly

Fine-tuning adapts a pretrained representation to a narrower distribution. Full fine-tuning updates all parameters and uses more optimizer memory; freezing a vision tower or training a projection/head is cheaper but can bottleneck adaptation. LoRA (Low-Rank Adaptation) freezes a base matrix W and learns a low-rank update `delta_W = (alpha/r) * B @ A`. It reduces trainable parameters, not necessarily inference latency unless merged or supported efficiently. LoRA is not the same as quantization.

This notebook uses a random linear layer as a transparent stand-in for a pretrained layer; it does not fine-tune an actual foundation checkpoint. The module and gradient updates are native PyTorch.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd

class LoRALinear(nn.Module):
    def __init__(self, in_features, out_features, rank=2, alpha=4.0):
        super().__init__()
        self.base = nn.Linear(in_features, out_features)
        self.base.weight.requires_grad_(False)
        self.base.bias.requires_grad_(False)
        self.A = nn.Parameter(torch.empty(rank, in_features))
        self.B = nn.Parameter(torch.zeros(out_features, rank))
        self.scale = alpha / rank
        nn.init.kaiming_uniform_(self.A, a=np.sqrt(5))

    def forward(self, x):
        update = F.linear(F.linear(x, self.A), self.B) * self.scale
        return self.base(x) + update

layer = LoRALinear(32, 24, rank=2)
trainable = sum(p.numel() for p in layer.parameters() if p.requires_grad)
total = sum(p.numel() for p in layer.parameters())
print(f'trainable={trainable} total={total} ratio={trainable / total:.1%}')
assert layer.B.count_nonzero() == 0  # initial adapter preserves base outputs

## Manual adapter training loop

We create a target task requiring a rank-1 correction to a frozen base layer. The example demonstrates the optimization pattern; real VLM fine-tuning typically injects adapters into selected attention and/or MLP projections, while deciding carefully whether to update the vision tower, multimodal projector, and language model.

In [ ]:
torch.manual_seed(18)
layer = LoRALinear(16, 12, rank=2, alpha=4.0)
x = torch.randn(512, 16)
u, v = torch.randn(12, 1), torch.randn(1, 16)
with torch.no_grad():
    y_target = layer.base(x) + 0.7 * F.linear(F.linear(x, v), u)
optimizer = torch.optim.AdamW([layer.A, layer.B], lr=2e-2)
history = []
for step in range(201):
    optimizer.zero_grad(set_to_none=True)
    prediction = layer(x)
    loss = F.mse_loss(prediction, y_target)
    loss.backward()
    optimizer.step()
    if step % 50 == 0:
        history.append({'step': step, 'mse': float(loss.detach())})
pd.DataFrame(history)

## Fine-tuning recipe and failure modes

1. Preserve a held-out set grouped by episode/source, not a random frame split.
2. Establish a frozen-base baseline; check image preprocessing and tokenizer/label alignment.
3. Train only intended modules; print trainable parameter names and count.
4. Use a conservative learning rate, gradient clipping, validation by task slice, and early stopping.
5. Compare LoRA against frozen head/projector and full fine-tuning when compute permits.
6. Keep a base checkpoint and test for catastrophic forgetting on general visual/language tasks.

For instruction tuning, labels usually supervise the assistant response rather than the user prompt; image placeholder tokens and multimodal sequence layout are model-specific. Never assume a generic label mask is correct for an architecture. Use a tiny overfit batch first to detect shifts, wrong EOS IDs, padding leakage, and frozen parameters with missing gradients.